In [1]:
import torch
import torch.nn as nn
import os
# 🌟 新增引入顶级流水线
from main_model import EndToEndDrivingPipeline, FrontViewSwinTiny
from CrosAttention import TemporalCrossAttention
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from dataset_create import prepare_multiple_datasets_and_scaler, ProcessedDrivingDataset, inverse_transform

D:\anaconda\envs\gzx\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class AutomaticWeightedLoss(nn.Module):
    """
    基于同方差不确定性的多任务动态 Loss 自适应权重权重分配
    """
    def __init__(self, num_tasks=2):
        super(AutomaticWeightedLoss, self).__init__()
        # 针对每个任务定义一个可学习的参数 log(sigma^2)，初始为 0
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, loss_velocity, loss_steer):
        log_var_vel = self.log_vars[0]
        log_var_steer = self.log_vars[1]

        precision_vel = torch.exp(-log_var_vel)
        precision_steer = torch.exp(-log_var_steer)

        # 动态加权 Loss：精度 * 误差 + 正则化项
        loss = (precision_vel * loss_velocity + log_var_vel) + \
               (precision_steer * loss_steer + log_var_steer)
        return loss

## 训练设定

In [3]:
def train(input_files, output_csv = 'processed_data.csv', scaler_json_path = 'scaler_params.json'):

    target_names = ['velocity', 'steer']    

    # 数据预处理
    print(">>> 开始检查并预处理数据...")
    scaler_params = prepare_multiple_datasets_and_scaler(
        input_files=input_files,
        output_csv=output_csv,
        scaler_json_path=scaler_json_path,
        seq_length=9)
    print(">>> 预处理完成！\n")

    # 加载数据集
    dataset = ProcessedDrivingDataset(csv_file=output_csv)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

    # 初始化模型、损失函数、优化器
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    unet_weight_path = r'E:\Laboratory files\code_project\D2D\visual\Segmentation\dpai_unet\dpai_unet.pth'
    
    model = EndToEndDrivingPipeline(unet_weight_path=unet_weight_path, 
        device=device, output_dim=2).to(device)# 因你的预测只有速度[0]和转角[1]两个数，设为2这极为关键！

    awl = AutomaticWeightedLoss(num_tasks=2).to(device)

    optimizer = Adam([
        {'params': model.parameters(), 'lr': 1e-4},
        {'params': awl.parameters(), 'lr': 1e-3}
    ])
    scheduler = StepLR(optimizer, step_size=10, gamma=0.9)
    criterion_huber = nn.HuberLoss(delta=0.1)

    # 设置加权损失的权重超参数
    Loss_name = 'HuberLoss'

    # 日志文件
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f"{Loss_name}.txt")

    # 训练循环
    num_epochs = 20
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_v_loss = 0.0
        running_s_loss = 0.0

        for batch_idx, (front_imgs, side_img, state_seq, target) in enumerate(dataloader):
            # 数据解包并推上显存
            front_imgs = [img.to(device) for img in front_imgs]
            side_img = side_img.to(device)
            state_seq = state_seq.to(device)
            target = target.to(device)
            # 🌟 第三重修改：前向传播！一针入魂：(3张纯前向RGB组, 1张纯左侧RGB, 8帧10维历史流)
            predictions = model(front_imgs, side_img, state_seq)

            batch_v_loss = criterion_huber(predictions[:, 0], target[:, 0])
            batch_s_loss = criterion_huber(predictions[:, 1], target[:, 1])
            loss = awl(batch_v_loss, batch_s_loss)
            # 反向传播与优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 累加损失用于打印
            running_loss += loss.item()*1000
            running_v_loss += batch_v_loss.item()*1000
            running_s_loss += batch_s_loss.item()*1000

            # 打印当前 Batch 的损失
            if (batch_idx + 1) % 5 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(dataloader)}], "
                      f"Total Loss: {loss.item():.6f}, V-Loss: {batch_v_loss.item():.10f}, S-Loss: {batch_s_loss.item():.10f}")

        # 学习率调度
        scheduler.step()

        # 记录日志
        avg_loss = running_loss / len(dataloader)
        avg_v = running_v_loss / len(dataloader)
        avg_s = running_s_loss / len(dataloader)
        
        with open(log_file, "a") as f:
            f.write(f"Epoch {epoch+1}: Total={avg_loss:.6f}, Vel={avg_v:.6f}, Steer={avg_s:.6f}\n")
        print(f"--- Epoch {epoch+1} 完成! 平均 Loss: {avg_loss:.6f} ---")
        # 🌟 修正4：定期保存模型
        if (epoch + 1) % 5 == 0:
            save_path = os.path.join(log_dir, f"DPAI_Driving_Epoch{epoch+1}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'awl_state_dict': awl.state_dict(),
            }, save_path)
            print(f">>> 模型 checkpoint 已保存至 {save_path}")
    print(">>> 训练全部完成！")

## 测试代码，用于观察其中维度

In [4]:

# 测试代码
if __name__ == '__main__':
    batch_size = 2
    embed_dim=384,    # 必须与 Swin 的输出通道一致
    num_patches=196,  # 对应 14x14 的 Patch 数量

    # 设定重要的自动驾驶类别权重
    my_weights = {0: 1.0, 1: 1.2, 7: 8.0, 11: 20.0}
    num_classes = 20

    # ==========================================
    # 1. 测试前端特征提取 (模拟真实图片输入)
    # ==========================================
    model = FrontViewSwinTiny(num_classes=num_classes, weights_dict=my_weights)

    # 生成假图像和掩码送入 Swin
    dummy_rgb = torch.randn(batch_size, 3, 224, 224)
    dummy_mask = torch.randint(0, num_classes, (batch_size, 1, 224, 224)).float()

    # 获取 Stage 4 的真实输出
    real_stage4_feat = model(dummy_rgb, dummy_mask)
    print(f"Swin Stage 4 真实输出形状: {real_stage4_feat.shape}")
    # 期望输出: [2, 49, 768]

    # ==========================================
    # 2. 测试时空交叉注意力融合
    # ==========================================
    fusion_module = TemporalCrossAttention(
        embed_dim=384,    # 必须与 Swin 的输出通道一致
        num_heads=12,     # 注意力头数，建议能被 embed_dim 整除
        num_patches=196,  # 对应 14x14 的 Patch 数量
        ffn_dim=1536      # 前馈网络的隐藏层维度
    )
    print(f"batch_size: {batch_size}, type: {type(batch_size)}")
    print(f"num_patches: {num_patches}, type: {type(num_patches)}")
    print(f"embed_dim: {embed_dim}, type: {type(embed_dim)}")
    # 修正方案：确保它们都是整数
    # 如果 num_patches 是 (14, 14)，你需要把它乘起来变成 196
    if isinstance(num_patches, tuple):
        num_patches = num_patches[0]
    # 确保 embed_dim 也是整数
    if isinstance(embed_dim, tuple):
        embed_dim = embed_dim[0]
    # 最终修正后的逻辑
    if isinstance(num_patches, tuple):
        # 如果元组里只有一个元素（如 (196,)），取索引 0
        # 如果有两个元素（如 (14, 14)），则相乘
        if len(num_patches) == 1:
            num_patches = num_patches[0]
        else:
            num_patches = num_patches[0] * num_patches[1]
    if isinstance(embed_dim, tuple):
        # 同样，对于 (384,) 取第一个元素
        embed_dim = embed_dim[0]
    # 现在定义随机张量就不会报错了

    # 模拟三帧 Stage 4 的输出 (当前帧 t, 过去帧 t1, t2)
    img_t = torch.randn(batch_size, num_patches, embed_dim)
    img_t1 = torch.randn(batch_size, num_patches, embed_dim)
    img_t2 = torch.randn(batch_size, num_patches, embed_dim)

    # 执行时空融合
    enhanced_feature = fusion_module(img_t, img_t1, img_t2)

    print(f"融合后的时空特征张量形状: {enhanced_feature.shape}")
    # 期望输出: [2, 49, 768]

[Debug Swin] FrontView Swin Output (Flattened): torch.Size([2, 196, 384])
Swin Stage 4 真实输出形状: torch.Size([2, 196, 384])
batch_size: 2, type: <class 'int'>
num_patches: (196,), type: <class 'tuple'>
embed_dim: (384,), type: <class 'tuple'>
融合后的时空特征张量形状: torch.Size([2, 196, 384])


## 开始训练代码

In [ ]:
# 找到你的原始数据相对于本 Notebook 的路径
csv_files = [
    'carla_data_collect/20260327_205825/csv/global_vehicle_data.csv'
    # 如果有更多文件，可以继续添加这里
]
# 启动训练
train(input_files=csv_files)

>>> 开始检查并预处理数据...
successfully
>>> 预处理完成！

✅ 预训练权重装填成功！
>>> 正在加载前向视角 Swin-Tiny 预训练权重...


D:\anaconda\envs\gzx\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\anaconda\envs\gzx\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


>>> 前向视角权重适配完成！
>>> 正在加载侧向视角 ResNet50 预训练权重...
>>> 侧向视角权重适配完成！
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): torch.Size([8, 196, 384])
[Debug Swin] FrontView Swin Output (Flattened): t